In [1]:
import pandas as pd

print("DE1")
file_path = "../data/raw/online_retail_ii.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

    
print("Total Rows:" , len(df))
print("Columns: ", list(df.columns))
print("\nMissing values per column:")
print(df.isnull().sum())

DE1
Total Rows: 1067371
Columns:  ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [ ]:
print("DE2")

initial_len = len(df)

#Drop missing values for Customer ID
df = df.dropna(subset=['Customer ID'])

missing_customer_id = initial_len - len(df)

print("Removed rows:", missing_customer_id)
print("Total Rows:" , len(df))
print("Columns: ", list(df.columns))
print("\nMissing values per column:")
print(df.isnull().sum())

DE2
Removed rows: 243007
Total rows: 824,364
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Missing values per column:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [ ]:
length_before_cancel = len(df)

df["Invoice"] = df["Invoice"].astype(str)
df = df[~df["Invoice"].str.contains("C", na=False)]

print("Removed rows:", length_before_cancel - len(df))
print("Total Rows:" , len(df))


Removed rows: 18744
Total rows: 805,620
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Missing values per column:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [4]:
length_before_negative = len(df)

df = df[(df["Quantity"] > 0) & (df["Price"] > 0)] 

print("Removed rows:", length_before_negative - len(df))
print("Total Rows:" , len(df))

Removed rows: 71
Total Rows: 805549


In [7]:

print(df["StockCode"].value_counts())

StockCode
85123A    5188
22423     3428
85099B    3360
84879     2777
20725     2678
          ... 
84630        1
79301        1
84567        1
20882        1
23843        1
Name: count, Length: 4631, dtype: int64


In [8]:
length_before_code = len(df)

non_product_codes=["POST", "M", "D", "BANK CHARGES"]

df = df[~df["StockCode"].isin(non_product_codes)]

print("Removed rows:", length_before_code - len(df))
print("Total Rows:" , len(df))

Removed rows: 2584
Total Rows: 802965


In [9]:
print("Total Rows after cleaning:" , len(df))

Total Rows after cleaning: 802965


In [12]:
print("DE3")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
max_date = df["InvoiceDate"].max()

test_split= max_date - pd.DateOffset(months=2)

train = df[df["InvoiceDate"] <= test_split]
test = df[df["InvoiceDate"] > test_split]

print("Train Rows:" , len(train))
print("Test Rows:" , len(test))

DE3
Train Rows: 684357
Test Rows: 118608


In [13]:
# DE4: sparse customer x product matrix from TRAINING rows only
import scipy.sparse as sp
import numpy as np

cust = {c: i for i, c in enumerate(train["Customer ID"].unique())}
prod = {p: j for j, p in enumerate(train["StockCode"].unique())}
rows = train["Customer ID"].map(cust); cols = train["StockCode"].map(prod)
vals = np.log1p(train["Quantity"]) # one possible purchase value
A = sp.csr_matrix((vals, (rows, cols)), shape=(len(cust), len(prod)))
print(A.shape, A.nnz, f"sparsity {1 - A.nnz / (A.shape[0] * A.shape[1]):.2%}")

(5484, 4537) 417809 sparsity 98.32%


In [14]:
import pickle

sp.save_npz("../data/processed/train_matrix.npz", A)

with open("../data/processed/cust_map.pkl", "wb") as f:
    pickle.dump(cust, f)
with open("../data/processed/prod_map.pkl", "wb") as f:
    pickle.dump(prod, f)

In [15]:
train.to_csv("../data/processed/train.csv", index=False)

In [16]:
test.to_csv("../data/processed/test.csv", index=False)